# PatchTST CI vs CD -- ECL

Trains channel-independent (CI) and channel-dependent (CD) variants of PatchTST at
pred_len in [96, 336] on ECL (321 variates, hourly) with identical hyperparameters and seed.
Results are saved to `results/results_ecl.csv`.

Split: 15840 train / 5256 val / 5256 test rows (Liu et al., iTransformer, ICLR 2024).

Memory note: the CD encoder receives C*N = 321*11 = 3531 tokens at seq_len=96. Fused
scaled-dot-product attention does not materialise the full (C*N)^2 score matrix, so
measured peak allocation (8.13 GiB at batch 128, 0.53 GiB at batch 8) sits well under a
T4's 16 GiB budget. This notebook trains CD at batch_size_cd=8 and CI at batch_size_ci=8,
matching effective batch size across modes to avoid a step-count confound.


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import math
import random
import time
from pathlib import Path
import shutil

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

SEED = 456
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Bump whenever the model architecture changes in a way that makes old
# checkpoints or results rows invalid to reuse (different state_dict keys,
# different forward-pass semantics, etc.) -- not for hyperparameter changes
# alone. Stamped into every checkpoint and every results row so a prior
# session output attached as an input can be checked before being
# trusted, rather than silently resumed or skipped as already-complete.
ARCH_VERSION = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"
print(f"Device: {DEVICE}")

## Dataset

In [ ]:
class ECLDataset(Dataset):
    """
    ECL (Electricity Consuming Load) multivariate dataset.

    321 client-level hourly electricity consumption variates.
    Split: ~60/20/20 proportional, matching iTransformer paper fractions (15840/5256/5256 out of 26352).
    Applied dynamically to handle version differences in the source file.

    Normalization: z-score per channel, scaler fit on train split only.
    Source: thuml/iTransformer repo, datasets/electricity.csv. Split matches Liu et al., iTransformer, ICLR 2024.
    Args:
        csv_path: Path to electricity.csv.
        split: One of 'train', 'val', 'test'.
        seq_len: Number of input timesteps per sample.
        pred_len: Number of target timesteps immediately following the input.
    """


    def __init__(self, csv_path: str, split: str, seq_len: int, pred_len: int) -> None:
        if split not in ("train", "val", "test"):
            raise ValueError(f"split must be one of 'train', 'val', 'test', got '{split}'.")

        df = pd.read_csv(csv_path, usecols=lambda c: c != "date")
        n = len(df)

        # Match iTransformer paper proportions (15840/5256/5256 out of 26352).
        # Applied to the actual row count to handle version differences.
        train_end = int(n * 0.6012)
        val_end = train_end + int(n * 0.1995)

        train_df = df.iloc[:train_end]
        self._mean = train_df.mean(axis=0).to_numpy(dtype=np.float32)
        self._std = train_df.std(axis=0, ddof=0).clip(lower=1e-8).to_numpy(dtype=np.float32)
        normalized = (df.values.astype(np.float32) - self._mean) / self._std

        if split == "train":
            self._data = normalized[:train_end]
        elif split == "val":
            self._data = normalized[train_end:val_end]
        else:
            self._data = normalized[val_end:]

        self.seq_len = seq_len
        self.pred_len = pred_len
        print(f"ECLDataset [{split}]: {len(self._data)} rows from {n} total (train_end={train_end}, val_end={val_end})")

    def __len__(self) -> int:
        return len(self._data) - self.seq_len - self.pred_len + 1

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        x = self._data[idx : idx + self.seq_len]
        y = self._data[idx + self.seq_len : idx + self.seq_len + self.pred_len]
        return torch.from_numpy(x), torch.from_numpy(y)


## Model

In [ ]:
class PatchEmbedding(nn.Module):
    """Linear patch projection with dropout and no positional encoding.

    Matches the paper's committed architecture: positional information is
    not added, which is part of the flattened-token CD behaviour under
    study.
    """

    def __init__(self, patch_size: int, d_model: int, dropout: float) -> None:
        super().__init__()
        self.proj = nn.Linear(patch_size, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.proj(x))


def _build_encoder(d_model: int, num_heads: int, num_layers: int, dropout: float) -> nn.TransformerEncoder:
    """Post-norm Transformer encoder, matching the paper's committed models.py exactly.

    Built from nn.TransformerEncoderLayer rather than a hand-rolled
    MultiheadAttention call: PyTorch's TransformerEncoderLayer dispatches
    its internal self-attention with need_weights=False, which enables the
    fused scaled-dot-product-attention kernel and avoids materialising the
    full (C*N)x(C*N) attention weight matrix. A hand-rolled equivalent that
    omits need_weights=False silently falls back to the unfused path and
    OOMs at this token count.
    """
    layer = nn.TransformerEncoderLayer(
        d_model=d_model,
        nhead=num_heads,
        dim_feedforward=d_model * 4,
        dropout=dropout,
        batch_first=True,
    )
    return nn.TransformerEncoder(layer, num_layers=num_layers)


class ForecastHead(nn.Module):
    def __init__(self, num_patches: int, d_model: int, pred_len: int) -> None:
        super().__init__()
        self.linear = nn.Linear(num_patches * d_model, pred_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(x.flatten(1))


class PatchTST(nn.Module):
    """PatchTST with CI and CD mode support.

    channel_mixing=False (CI): each variate processed independently; encoder sees N patch tokens per channel with weights shared across channels by construction.
    channel_mixing=True  (CD): patches from all variates concatenated along the sequence dimension before the encoder; attention runs over C*N tokens simultaneously.

    Reference: Nie et al., "A Time Series Is Worth 64 Words", ICLR 2023.
               https://arxiv.org/abs/2211.14730
    """

    def __init__(self, seq_len: int, pred_len: int, num_variates: int, patch_size: int = 16, stride: int = 8, d_model: int = 128, num_heads: int = 16,
                 num_layers: int = 3, dropout: float = 0.2, channel_mixing: bool = False) -> None:
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError(f"d_model ({d_model}) must be divisible by num_heads ({num_heads}).")
        self.patch_size = patch_size
        self.stride = stride
        self.num_variates = num_variates
        self.channel_mixing = channel_mixing
        self.num_patches = (seq_len - patch_size) // stride + 1
        self.embedding = PatchEmbedding(patch_size=patch_size, d_model=d_model, dropout=dropout)
        self.encoder = _build_encoder(d_model=d_model, num_heads=num_heads, num_layers=num_layers, dropout=dropout)
        self.head = ForecastHead(num_patches=self.num_patches, d_model=d_model, pred_len=pred_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, C = x.shape
        flat = x.permute(0, 2, 1).reshape(B * C, L)
        patches = flat.unfold(-1, self.patch_size, self.stride)  # (B*C, N, P)
        x = self.embedding(patches)  # (B*C, N, D)
        if self.channel_mixing:
            x = x.reshape(B, C * self.num_patches, -1)  # (B, C*N, D)
            x = self.encoder(x)                          # (B, C*N, D)
            x = x.reshape(B * C, self.num_patches, -1)  # (B*C, N, D)
        else:
            x = self.encoder(x)  # (B*C, N, D)
        x = self.head(x)  # (B*C, pred_len)
        return x.reshape(B, C, -1).permute(0, 2, 1)  # (B, pred_len, C)


## Training Infrastructure

In [ ]:
# -- Per-epoch atomic checkpoint + resume, ported from the boundary-sweep
# notebooks (train_boundary_p4_temp*.ipynb). Same invariant: the file on disk
# always describes a state at an epoch BOUNDARY (written after val + early-
# stop bookkeeping). A resumed session restores parameters, optimiser,
# scaler, RNG streams, and early-stop state, then continues with the next
# epoch, so a session killed mid-run loses at most the in-progress epoch.

def _atomic_torch_save(obj: dict, path: Path) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def _atomic_csv_save(rows: list[dict], path: Path) -> None:
    tmp = path.with_suffix(".csv.tmp")
    pd.DataFrame(rows).to_csv(tmp, index=False)
    os.replace(tmp, path)


def _rng_capture() -> dict:
    return {
        "py": random.getstate(),
        "np": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }


def _rng_restore(st: dict) -> None:
    random.setstate(st["py"])
    np.random.set_state(st["np"])
    torch.set_rng_state(st["torch"].cpu() if torch.is_tensor(st["torch"]) else st["torch"])
    if st["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([t.cpu() if torch.is_tensor(t) else t for t in st["cuda"]])


class EarlyStopping:
    """Stop training when validation MSE stops improving.

    Resumable: `state_dict()` / `load_state_dict()` capture everything needed
    to continue mid-training after a session restart, matching the boundary
    notebooks' resume contract. The best model weights are checkpointed
    separately as part of the outer per-epoch save (see `run_mode`), not by
    this class directly.

    Args:
        patience: Number of epochs to wait after last improvement.
    """

    def __init__(self, patience: int = 10) -> None:
        self.patience = patience
        self.best_val_mse = float("inf")
        self.counter = 0
        self.best_epoch = 0

    def step(self, val_mse: float, epoch: int) -> bool:
        """Update best/counter for the current epoch; return True if patience exhausted."""
        improved = val_mse < self.best_val_mse
        if improved:
            self.best_val_mse = val_mse
            self.counter = 0
            self.best_epoch = epoch
        else:
            self.counter += 1
        return self.counter >= self.patience

    def state_dict(self) -> dict:
        return {
            "patience": self.patience,
            "best_val_mse": self.best_val_mse,
            "counter": self.counter,
            "best_epoch": self.best_epoch,
        }

    def load_state_dict(self, st: dict) -> None:
        self.patience = st["patience"]
        self.best_val_mse = st["best_val_mse"]
        self.counter = st["counter"]
        self.best_epoch = st["best_epoch"]


def compute_metrics(pred: torch.Tensor, target: torch.Tensor) -> tuple[float, float]:
    """Compute MSE and MAE between predictions and targets.

    Args:
        pred: Predicted tensor, any shape.
        target: Ground truth tensor, same shape as pred.

    Returns:
        Tuple of (mse, mae) as Python floats.
    """
    mse = torch.mean((pred - target) ** 2).item()
    mae = torch.mean(torch.abs(pred - target)).item()
    return mse, mae


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    scaler: torch.amp.GradScaler,
    accum_steps: int = 1,
) -> tuple[float, float]:
    """Run one training epoch with gradient accumulation and FP16 autocast.

    Args:
        model: The model to train.
        loader: Training DataLoader.
        optimizer: Optimizer instance.
        criterion: Loss function (MSELoss).
        scaler: GradScaler for FP16 mixed precision.
        accum_steps: Number of steps to accumulate gradients over before updating.

    Returns:
        Tuple of (mean_mse, mean_mae) weighted by batch size.
    """
    model.train()
    total_mse, total_mae, n = 0.0, 0.0, 0
    optimizer.zero_grad()
    for step, (x, y) in enumerate(loader):
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.amp.autocast(device_type=DEVICE.type, dtype=torch.float16):
            pred = model(x)
            loss = criterion(pred, y) / accum_steps
        scaler.scale(loss).backward()
        if (step + 1) % accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        mae = torch.mean(torch.abs(pred.detach().float() - y.float())).item()
        batch = x.size(0)
        total_mse += loss.item() * accum_steps * batch
        total_mae += mae * batch
        n += batch
    return total_mse / n, total_mae / n


@torch.inference_mode()
def evaluate(model: nn.Module, loader: DataLoader) -> tuple[float, float]:
    """Evaluate model on a DataLoader and return mean MSE and MAE.

    Args:
        model: The model to evaluate.
        loader: Validation or test DataLoader.

    Returns:
        Tuple of (mean_mse, mean_mae) weighted by batch size.
    """
    model.eval()
    total_mse, total_mae, n = 0.0, 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.amp.autocast(device_type=DEVICE.type, dtype=torch.float16):
            pred = model(x)
        mse, mae = compute_metrics(pred.float(), y.float())
        batch = x.size(0)
        total_mse += mse * batch
        total_mae += mae * batch
        n += batch
    return total_mse / n, total_mae / n

## run_mode

In [ ]:
def run_mode(mode: str, csv_path: str, config: dict, results_dir: Path, ckpt_dir: Path,
             session_t0: float, session_budget_s: float, epoch_est_s: dict) -> dict | None:
    """One CI/CD run for ECL, resumable at epoch granularity.

    Same contract as the boundary notebooks' `train_one()`: returns the
    result row on completion, or None if the session budget was reached
    before the run finished (the checkpoint stays on disk; the next
    session's call to this function resumes from it automatically).

    With seq_len=96, patch_size=16, stride=8: N=11 patches per variate.
    CD encoder receives C*N = 321*11 = 3531 tokens. Measured on a T4 at this
    architecture: 892.9 s/epoch at batch 128 (8.13 GiB peak), 894.9 s/epoch
    at batch 8 (0.53 GiB peak).

    Args:
        mode: 'CI' or 'CD'.
        csv_path: Path to electricity.csv.
        config: Dict containing all hyperparameters plus 'pred_len' and 'seed'.
        results_dir: Directory for output CSVs.
        ckpt_dir: Directory for model checkpoints.
        session_t0: time.time() at session start, for budget accounting.
        session_budget_s: Wall-clock budget for this Kaggle session, seconds.
        epoch_est_s: {"CD": est_seconds_per_epoch, "CI": est_seconds_per_epoch},
            refined as epochs complete; used to decide whether the next epoch
            can fit before the session budget.

    Returns:
        Result dict, or None if the session budget was reached mid-run.
    """
    channel_mixing = mode == "CD"
    pred_len = config["pred_len"]
    seed = config["seed"]
    batch_size = config["batch_size_cd"] if channel_mixing else config["batch_size_ci"]
    accum_steps = config["accum_steps_cd"] if channel_mixing else config["accum_steps_ci"]
    effective_batch = batch_size * accum_steps
    ckpt_path = ckpt_dir / f"ckpt_ecl_{mode.lower()}_pred{pred_len}_s{seed}.pt"

    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)

    # Dataset objects are cheap metadata (steps_per_epoch needs their length);
    # DataLoaders that spin up persistent worker processes are deferred until
    # we know they're actually needed. A resumed run that already early-
    # stopped needs only test_loader for the final eval -- building train/val
    # DataLoaders (2 workers each, persistent) for a run that will never
    # iterate them wastes real wall-clock and process overhead on every
    # already-finished run touched by a fresh session.
    train_ds = ECLDataset(csv_path, "train", config["seq_len"], pred_len)
    val_ds = ECLDataset(csv_path, "val", config["seq_len"], pred_len)
    test_ds = ECLDataset(csv_path, "test", config["seq_len"], pred_len)

    loader_kwargs: dict = {
        "batch_size": batch_size,
        "num_workers": 2,
        "persistent_workers": True,
        "pin_memory": PIN_MEMORY,
    }
    train_loader = None  # built lazily, see below
    val_loader = None    # built lazily, see below
    test_loader = DataLoader(test_ds, shuffle=False, **loader_kwargs)

    model = PatchTST(
        seq_len=config["seq_len"], pred_len=pred_len, num_variates=config["num_variates"],
        patch_size=config["patch_size"], stride=config["stride"], d_model=config["d_model"],
        num_heads=config["num_heads"], num_layers=config["num_layers"], dropout=config["dropout"],
        channel_mixing=channel_mixing,
    ).to(DEVICE)

    num_patches = model.num_patches
    total_params = sum(p.numel() for p in model.parameters())
    steps_per_epoch = math.ceil(len(train_ds) / batch_size)
    effective_steps_per_epoch = math.ceil(steps_per_epoch / accum_steps)

    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=1e-4)
    warmup_epochs = config["warmup_epochs"]

    def lr_lambda(epoch: int) -> float:
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, config["epochs"] - warmup_epochs)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    criterion = nn.MSELoss()
    scaler = torch.amp.GradScaler(device="cuda")
    early_stopping = EarlyStopping(patience=config["patience"])

    start_epoch = 1
    stopped = False
    best_state_dict = None

    if ckpt_path.exists():
        ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        ckpt_arch = ckpt.get("arch_version")
        if ckpt_arch != ARCH_VERSION:
            raise RuntimeError(
                f"{ckpt_path.name} was written by arch_version={ckpt_arch!r}, "
                f"current is {ARCH_VERSION}. Loading its weights into the current "
                f"model would silently mismatch or produce results from a different "
                f"architecture. Delete this checkpoint and any results_ecl.csv rows "
                f"for this (mode, pred_len, seed) and rerun from scratch."
            )
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        scaler.load_state_dict(ckpt["scaler"])
        early_stopping.load_state_dict(ckpt["early_stopping"])
        _rng_restore(ckpt["rng"])
        best_state_dict = ckpt["best_model_state"]
        start_epoch = ckpt["epoch_done"] + 1
        stopped = ckpt["stopped"]
        print(f"  [resume] {ckpt_path.name}: {ckpt['epoch_done']} epochs done, "
              f"best_val={early_stopping.best_val_mse:.6f} @ epoch {early_stopping.best_epoch}, "
              f"stopped={stopped}")

    print(
        f"[{mode} pred={pred_len} seed={seed}] params: {total_params:,} | batch: {batch_size} | "
        f"accum: {accum_steps} | effective_batch: {effective_batch} | "
        f"steps/epoch: {steps_per_epoch} | effective_steps/epoch: {effective_steps_per_epoch} | "
        f"CD_tokens: {config['num_variates'] * num_patches if channel_mixing else 'N/A'} | "
        f"start_epoch: {start_epoch}"
    )

    t0 = time.time()
    if not stopped:
        train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
        val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
        for epoch in range(start_epoch, config["epochs"] + 1):
            elapsed = time.time() - session_t0
            est = epoch_est_s[mode]
            if elapsed + est + _BUDGET_MARGIN_S > session_budget_s:
                print(f"  [{mode} pred={pred_len} seed={seed}] Session budget reached before epoch "
                      f"{epoch}. Checkpoint saved; next session resumes here.")
                _atomic_torch_save({
                    "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                    "early_stopping": early_stopping.state_dict(), "rng": _rng_capture(),
                    "best_model_state": best_state_dict, "epoch_done": epoch - 1, "stopped": False,
                    "arch_version": ARCH_VERSION,
                }, ckpt_path)
                del model, optimizer, scheduler, criterion, scaler, train_loader, val_loader, test_loader
                del train_ds, val_ds, test_ds
                if DEVICE.type == "cuda":
                    torch.cuda.empty_cache()
                return None

            ep_t0 = time.time()
            train_mse, _ = train_one_epoch(model, train_loader, optimizer, criterion, scaler, accum_steps)
            val_mse, _ = evaluate(model, val_loader)
            scheduler.step()
            ep_s = time.time() - ep_t0
            epoch_est_s[mode] = max(epoch_est_s[mode], ep_s)  # tighten estimate from real measurement

            improved = val_mse < early_stopping.best_val_mse
            stop_now = early_stopping.step(val_mse, epoch)
            if improved:
                best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}

            if epoch % 5 == 0 or epoch == 1 or stop_now:
                lr_now = optimizer.param_groups[0]["lr"]
                print(
                    f"  [{mode} pred={pred_len} seed={seed}] Epoch {epoch:3d}/{config['epochs']} | "
                    f"train MSE {train_mse:.4f} | val MSE {val_mse:.4f} | "
                    f"lr {lr_now:.2e} | {ep_s:.0f}s/ep | {time.time() - t0:.0f}s total"
                )

            _atomic_torch_save({
                "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                "early_stopping": early_stopping.state_dict(), "rng": _rng_capture(),
                "best_model_state": best_state_dict, "epoch_done": epoch, "stopped": stop_now,
                "arch_version": ARCH_VERSION,
            }, ckpt_path)

            if stop_now:
                print(f"  [{mode} pred={pred_len} seed={seed}] Early stop @ epoch {epoch}. "
                      f"Best: epoch {early_stopping.best_epoch}.")
                break

    assert best_state_dict is not None, (
        f"[{mode} pred={pred_len} seed={seed}] No improving epoch was ever recorded "
        f"before this point -- likely SESSION_BUDGET_S is too small to fit even one "
        f"epoch plus margin. Increase SESSION_BUDGET_S and resume."
    )
    model.load_state_dict(best_state_dict)
    test_mse, test_mae = evaluate(model, test_loader)
    print(
        f"  [{mode} pred={pred_len} seed={seed}] Test MSE: {test_mse:.4f} | Test MAE: {test_mae:.4f} | "
        f"Best val MSE: {early_stopping.best_val_mse:.4f} @ epoch {early_stopping.best_epoch}"
    )

    del model, optimizer, scheduler, criterion, scaler
    del train_loader, val_loader, test_loader
    del train_ds, val_ds, test_ds
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return {
        "mode": mode, "pred_len": pred_len, "seed": seed,
        "test_mse": round(test_mse, 6), "test_mae": round(test_mae, 6),
        "best_val_mse": round(early_stopping.best_val_mse, 6),
        "best_epoch": early_stopping.best_epoch,
        "num_params": total_params,
        "batch_size": batch_size, "accum_steps": accum_steps, "effective_batch": effective_batch,
        "steps_per_epoch": steps_per_epoch, "effective_steps_per_epoch": effective_steps_per_epoch,
        "arch_version": ARCH_VERSION,
    }


## Config

In [ ]:
CSV_PATH = "/kaggle/input/datasets/choikahou/tslib-demo-datasets/all_datasets/electricity/electricity.csv"
RESULTS_DIR = Path("results")
CKPT_DIR = Path("results/checkpoints")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# seq_len=96, 321 variates, architecture (d_model=128, 16 heads). CD fits
# comfortably at batch_size_cd=8 with no gradient accumulation (8.13 GiB
# peak, 892.9 s/epoch, measured on a T4). CI matches at batch_size_ci=8 so
# both modes train at the same effective batch size, avoiding a step-count
# confound between them.
BASE_CONFIG = {
    "seq_len": 96,
    "num_variates": 321,
    "patch_size": 16,
    "stride": 8,
    "d_model": 128,
    "num_heads": 16,
    "num_layers": 3,
    "dropout": 0.2,
    "lr": 1e-4,
    "warmup_epochs": 5,
    "batch_size_ci": 8,
    "accum_steps_ci": 1,
    "batch_size_cd": 8,
    "accum_steps_cd": 1,
    "epochs": 50,
    "patience": 10,
    "seed": SEED,
}

PRED_LENS = [96, 336]

# -- Session budget (12h Kaggle cap kills a run WITH NO SAVED OUTPUT if unmanaged) --
SESSION_T0 = time.time()

# REQUIRED, set before Run All: this session's actual remaining Kaggle quota,
# in seconds. No numeric default is provided on purpose -- a stale or generic
# value here can exceed Kaggle's 12h hard cap, which makes the budget check
# below unreachable and kills the session with no saved output.
# Example: SESSION_BUDGET_S = 2.2 * 3600  # for ~2h20m remaining
SESSION_BUDGET_S = None
assert SESSION_BUDGET_S is not None, (
    "SESSION_BUDGET_S is unset. Set it to this session's actual remaining "
    "Kaggle quota in seconds before Run All. Do not reuse a previous "
    "session's value or assume the full 12h cap is available."
)

_BUDGET_MARGIN_S = 900.0  # test eval + checkpoint + version-save overhead
# Conservative per-epoch seeds (892.9 s/ep CD @ b8 measured, ~90 s/ep CI @
# b8 estimated); refined in-place as real epochs complete within a session.
EPOCH_EST_S = {"CD": 950.0, "CI": 150.0}


## Run CI and CD

In [ ]:
WORK = RESULTS_DIR
results_path = WORK / "results_ecl.csv"

# -- Cross-session bootstrap: a fresh session's /kaggle/working starts empty.
# If the previous version's output is attached as an input, seed working
# state from it -- registry first, then any checkpoints -- before deciding
# what still needs to run. Files already in working win, so a same-session
# rerun never regresses to older state. Mirrors the boundary notebooks'
# Cell-6 bootstrap.
INPUT_ROOT = Path("/kaggle/input")
_prior_csvs = [p for p in INPUT_ROOT.rglob("results_ecl.csv")]
if not results_path.exists() and _prior_csvs:
    assert len(_prior_csvs) == 1, (
        f"Multiple prior registries attached: {_prior_csvs}. Detach all but the latest version's output.")
    shutil.copy(_prior_csvs[0], results_path)
    print(f"[bootstrap] registry seeded from {_prior_csvs[0]}")

_prior_ckpts: dict[str, list[Path]] = {}
for _p in INPUT_ROOT.rglob("ckpt_ecl_*.pt"):
    _prior_ckpts.setdefault(_p.name, []).append(_p)
for _name, _srcs in sorted(_prior_ckpts.items()):
    assert len(_srcs) == 1, (
        f"Checkpoint {_name} found in multiple attached inputs: {_srcs}. Detach all but the latest version's output.")
    if not (CKPT_DIR / _name).exists():
        _ckpt_probe = torch.load(_srcs[0], map_location="cpu", weights_only=False)
        _probe_arch = _ckpt_probe.get("arch_version")
        del _ckpt_probe
        if _probe_arch != ARCH_VERSION:
            print(f"[bootstrap] SKIPPING {_name}: arch_version={_probe_arch!r} != "
                  f"current {ARCH_VERSION}. Written by a different architecture -- "
                  f"not seeded, will retrain from scratch.")
            continue
        shutil.copy(_srcs[0], CKPT_DIR / _name)
        print(f"[bootstrap] {_name} seeded from {_srcs[0]}")

all_results = []
completed = set()
if results_path.exists() and results_path.stat().st_size > 0:
    existing = pd.read_csv(results_path)
    if "arch_version" in existing.columns:
        stale_mask = existing["arch_version"] != ARCH_VERSION
    else:
        stale_mask = pd.Series(True, index=existing.index)  # predates this safeguard entirely
    if stale_mask.any():
        print(f"[bootstrap] Dropping {stale_mask.sum()} row(s) from a different or missing "
              f"arch_version (current {ARCH_VERSION}) -- not trusted as complete, will retrain:")
        print(existing[stale_mask][["mode", "pred_len", "seed"]].to_string(index=False))
    existing = existing[~stale_mask]
    all_results = existing.to_dict("records")
    completed = {(r["mode"], int(r["pred_len"]), int(r["seed"])) for r in all_results}
    print(f"Registry: {len(completed)} runs already complete (current architecture): {completed}")
RUN_LIST = [(mode, pred_len) for pred_len in PRED_LENS for mode in ["CI", "CD"]]

for mode, pred_len in RUN_LIST:
    config = {**BASE_CONFIG, "pred_len": pred_len}
    key = (mode, pred_len, config["seed"])
    if key in completed:
        print(f"SKIP {mode} pred_len={pred_len} seed={config['seed']} (already in results)")
        continue
    print(f'\n{"=" * 60}')
    print(f"Mode: {mode} | pred_len: {pred_len} | seed: {config['seed']}")
    print(f'{"=" * 60}')
    result = run_mode(mode, CSV_PATH, config, RESULTS_DIR, CKPT_DIR,
                       SESSION_T0, SESSION_BUDGET_S, EPOCH_EST_S)
    if result is None:
        print("Session budget reached. Next session: new version, attach THIS version's "
              "output as an input, Run All to resume.")
        break
    all_results.append(result)
    completed.add(key)
    _atomic_csv_save(all_results, results_path)
    print(f"  Results written ({len(all_results)}/{len(RUN_LIST)} runs complete)")

results_df = pd.DataFrame(all_results)
print("\n=== CI vs CD Results -- ECL ===")
if not results_df.empty:
    print(results_df[["mode", "pred_len", "seed", "test_mse", "test_mae", "best_epoch"]].to_string(index=False))


## Verify Output Files

In [ ]:
expected_rows = len(PRED_LENS) * 2  # {CI, CD} x each pred_len, one seed
required = [RESULTS_DIR / "results_ecl.csv"]
if not all(p.exists() for p in required):
    missing = [p for p in required if not p.exists()]
    raise RuntimeError(f"Output files missing: {missing}. Do not close the session.")

final_df = pd.read_csv(RESULTS_DIR / "results_ecl.csv")
print(f"Rows in results_ecl.csv: {len(final_df)} (expected {expected_rows}: "
      f"{len(PRED_LENS)} pred_lens x {{CI, CD}}, seed={BASE_CONFIG['seed']})")
print(final_df[["mode", "pred_len", "seed", "test_mse", "test_mae", "best_epoch"]].to_string(index=False))
if len(final_df) < expected_rows:
    print("WARNING: not all runs complete -- Run All again to resume from checkpoint.")
else:
    print("All output files verified.")
